# LLM pipeline walkthrough

This notebook traces end-to-end how a 30-member DeepSeek-LLM-7B-Chat ensemble plus router is produced. It is a reading guide for `cka_ens.llm` rather than a runnable demo; running it end-to-end requires ~30 GPU-hours and is not exercised by CI.

Pipeline stages:
1. **Member training** (sequential, with CKA penalty against frozen anchors)
2. **Per-member benchmark evaluation** (produces `all_results.csv`)
3. **Embed prompts + pivot rewards** (produces `router_data_checkpoint.pt`)
4. **Router training** (BGE + residual MLP)
5. **Router evaluation** (top-1 deployment)

## 1. Sequential member training

The first member is a plain fine-tune; every subsequent member trains with an active CKA penalty against all previous members.

In [ ]:
!for i in $(seq 0 29); do \
    python -m cka_ens.llm.train \
        --root ./members_cka --idx $i --mode cka --lambda 10.0; \
done

Same command for the control (no penalty) and COS² baselines, with different `--mode` and a different root directory.

## 2. Per-member benchmark evaluation

Each member is scored on the six benchmarks (MMLU, GSM8K, HumanEval, ARC-C, ARC-E, GPQA). Rewards are 0/1; output is appended to `all_results.csv`.

In [ ]:
for bench in ["mmlu", "gsm8k", "humaneval", "arc_challenge", "arc_easy", "gpqa"]:
    !python -m cka_ens.llm.evaluate --root ./members_cka --benchmark {bench}

## 3. Embed + pivot for the router

`Router.ipynb` (or the script port below) does two things:
- BGE-encodes every unique question
- Pivots `all_results.csv` into a per-question reward matrix Y of shape `(num_questions, num_experts)`.

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

df = pd.read_csv("all_results.csv")
pivot = (df.pivot_table(
    index=["Question", "Benchmark"],
    columns="Model", values="Reward", aggfunc="mean")
  .fillna(0.0).reset_index())

texts = pivot["Question"].astype(str).tolist()
benchmarks = pivot["Benchmark"].tolist()
model_cols = sorted([c for c in pivot.columns if c.startswith("model_")])
Y = torch.tensor(pivot[model_cols].values, dtype=torch.float32)

embedder = SentenceTransformer("BAAI/bge-large-en-v1.5")
X = embedder.encode(texts, convert_to_tensor=True).cpu()

torch.save({"X": X, "Y_rewards": Y, "texts": texts,
            "benchmarks": benchmarks, "model_names": model_cols},
           "router_data_checkpoint.pt")

## 4. Router training

In [ ]:
!python -m cka_ens.llm.router_train \
    --checkpoint router_data_checkpoint.pt --out router.pt

## 5. Router evaluation

In [ ]:
!python -m cka_ens.llm.router_evaluate \
    --router router.pt --data router_data_checkpoint.pt --topk 1

Final numbers should reproduce Table 4 of the paper (±noise from a different random seed). The headline router-test-split row is:

| Benchmark | Best single | Router (top-1) | Oracle |
|---|---|---|---|
| MMLU | 49.3% | 55.1% | 71.0% |
| GSM8K | 96.8% | 98.4% | 100.0% |
| HumanEval | 50.0% | 56.3% | 81.3% |
| ARC-Challenge | 59.7% | 63.9% | 75.6% |
| ARC-Easy | 79.4% | 82.6% | 90.4% |